# TTS Dataset Selection

Step 5 (and again step 8) of the OpenVoiceOS TTS dataset pipeline
([pipeline.md](../docs/pipeline.md)). This notebook reads a manifest whose rows carry a
`scores` object ([manifest.md](../docs/manifest.md)) and a list of selection rules, and
writes a **selection file**: the `id` values to keep, plus a `.meta.json` sidecar. It never
scores a clip — that is the measure notebook's job — and it never deletes, moves or opens
any audio file.

A manifest can be selected more than once with different rules, without scoring again. Run
this notebook after the measure notebook on a donor or corpus manifest (step 5), and again
after voice conversion and its own measure pass, to select the clips that go into training
(step 8).

**Inputs**
- a manifest (JSON Lines, [manifest.md](../docs/manifest.md)) whose rows carry `scores`;
- a `RULES` list of selection rules (see the config cell and [manifest.md](../docs/manifest.md)).

**Outputs**
- a selection file: JSON Lines, one kept `id` value per line;
- its `<selection file>.meta.json` sidecar: every rule, the row count it removed, the input
  manifest's path and sha256, and the row count kept.

**Hardware.** CPU only, no network access, no model and no GPU. This notebook does not use
onnxruntime, so the `ONNX_PROVIDERS` and `ONNX_RUNTIME` variables that the synthesis, measure
and conversion notebooks read do not apply here.


---
## NGI0 Grant Attribution

> Developed by TigreGotico for OpenVoiceOS, funded by the
> [NGI0 Commons Fund](https://nlnet.nl/project/OpenVoiceOS) / NLnet grant **101135429**.
> European Commission Next Generation Internet programme — DG CONNECT,
> grant agreement No 101135429.
> Additional funding: Swiss State Secretariat for Education, Research and Innovation (SERI).

**NGI work item:** WP3 — TTS dataset selection
**CI execution:** executed by CI (no GPU, no heavy download; see the `skip_execute` note in
[README.md](../README.md))

---


## 0a &middot; Install dependencies

This notebook reads JSON and computes a few statistics, so it uses only the Python standard
library (`json`, `hashlib`, `math`, `pathlib`). It installs nothing.


In [ ]:
import sys

assert sys.version_info >= (3, 9), "This notebook needs Python 3.9 or newer."
print(f"Python {sys.version.split()[0]} \u2014 no external dependencies to install.")


## 0b &middot; Global configuration

All parameters are environment variables, set with `os.environ.setdefault()`. Override any
of them from your shell, a `.env` file, or `%env` magic **before** running this cell.

| Variable | Meaning |
|---|---|
| `MANIFEST_PATH` | Path to the scored input manifest (JSON Lines, [manifest.md](../docs/manifest.md)). Empty: a built-in 20-row demo manifest is written and used, so the notebook always has something to run on. |
| `RULES` | A JSON list of selection rules, applied in order. A row is removed by the **first** rule it fails. Default `[]`: keeps every row. |
| `MISSING` | `keep` or `drop` &mdash; what happens to a row that lacks the score or field a rule reads. Default `keep`. |
| `SELECTION_PATH` | Output selection file. Empty: derived from `MANIFEST_PATH` as `<stem>_selection.jsonl`. |
| `ALLOW_EMPTY` | `1` allows writing a selection that keeps zero rows. Default `0`: refuse and raise instead. |

### Rule shape

```json
{"score": "cer", "op": "<=", "value": 0.10}
{"field": "duration_s", "op": "between", "value": [1.0, 15.0]}
{"score": "len_diff_s", "op": "abs<=", "value": 0.07}
```

`score` reads a key inside the row's `scores` object; `field` reads a top-level row field
(for example `duration_s`). `op` is one of `<=`, `>=`, `<`, `>`, `between` (value is
`[low, high]`), or `abs<=` (compares the absolute value against `value`).


In [ ]:
import json
import os

os.environ.setdefault("MANIFEST_PATH", "")   # scored input manifest; empty = built-in demo
os.environ.setdefault("RULES", "[]")         # JSON list of rules, applied in order; [] keeps all
os.environ.setdefault("MISSING", "keep")     # keep | drop
os.environ.setdefault("SELECTION_PATH", "")  # empty = derive from MANIFEST_PATH
os.environ.setdefault("ALLOW_EMPTY", "0")    # 1 = allow a selection that keeps zero rows

MANIFEST_PATH = os.environ["MANIFEST_PATH"]
RULES = json.loads(os.environ["RULES"])
MISSING = os.environ["MISSING"]
SELECTION_PATH = os.environ["SELECTION_PATH"]
ALLOW_EMPTY = os.environ["ALLOW_EMPTY"].strip().lower() in ("1", "true", "yes")

if MISSING not in ("keep", "drop"):
    raise ValueError(f"MISSING must be 'keep' or 'drop', got {MISSING!r}")
if not isinstance(RULES, list):
    raise ValueError("RULES must be a JSON list of rule objects")

print("Configuration loaded.")
print(f"  MANIFEST_PATH  : {MANIFEST_PATH or '(unset — using the built-in demo manifest)'}")
print(f"  RULES          : {len(RULES)} rule(s)")
print(f"  MISSING        : {MISSING}")
print(f"  SELECTION_PATH : {SELECTION_PATH or '(unset — derived from the manifest path)'}")
print(f"  ALLOW_EMPTY    : {ALLOW_EMPTY}")


## 1 &middot; Selection functions

A rule is checked in order against every row still in play. A row that fails a rule is
removed there and its `id` never reaches a later rule. A row that lacks the key a rule reads
is counted on its own line, `missing <key>`, and handled according to `MISSING`: `keep`
lets it through to the next rule unaffected; `drop` removes it at that rule. Either way the
missing count is always printed &mdash; a missing score is never silently passed through and
never silently counted as a failure.

For every rule, the report below also shows the distribution (min, p5, p50, p95, max) of the
values the rule read, and how many of them sit within 5% of the rule's cut &mdash; a cut with
many rows in that band is landing inside a cluster, not in a gap.


In [ ]:
import hashlib
import math
from pathlib import Path


def read_manifest(path):
    """One JSON row per non-empty line, in file order."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def rule_key(rule):
    """The (key, is_score) a rule reads: is_score True for 'scores', False for a top-level field."""
    if "score" in rule:
        return rule["score"], True
    if "field" in rule:
        return rule["field"], False
    raise ValueError(f"rule has neither 'score' nor 'field': {rule}")


def rule_description(rule):
    key, is_score = rule_key(rule)
    kind = "score" if is_score else "field"
    return f"{kind} {key} {rule['op']} {rule['value']}"


def row_value(row, key, is_score):
    """(value, present). present is False when the row lacks the key."""
    container = row.get("scores", {}) if is_score else row
    if key not in container:
        return None, False
    return container[key], True


def rule_passes(value, op, target):
    if op == "<=":
        return value <= target
    if op == ">=":
        return value >= target
    if op == "<":
        return value < target
    if op == ">":
        return value > target
    if op == "between":
        lo, hi = target
        return lo <= value <= hi
    if op == "abs<=":
        return abs(value) <= target
    raise ValueError(f"unknown op: {op}")


def _cut_points(op, target):
    return list(target) if op == "between" else [target]


def near_cut(value, op, target, fraction=0.05):
    """True when value sits within `fraction` of one of the rule's cut points."""
    v = abs(value) if op == "abs<=" else value
    for cp in _cut_points(op, target):
        window = abs(cp) * fraction if cp else fraction
        if abs(v - cp) <= window:
            return True
    return False


def _percentile(sorted_values, p):
    if len(sorted_values) == 1:
        return sorted_values[0]
    k = (len(sorted_values) - 1) * (p / 100)
    f, c = math.floor(k), math.ceil(k)
    if f == c:
        return sorted_values[int(k)]
    return sorted_values[f] * (c - k) + sorted_values[c] * (k - f)


def distribution(values):
    if not values:
        return None
    s = sorted(values)
    return {"min": s[0], "p5": _percentile(s, 5), "p50": _percentile(s, 50),
            "p95": _percentile(s, 95), "max": s[-1]}


def select(rows, rules, missing_policy):
    """Apply rules in order. Returns (kept_ids, report); one report entry per rule."""
    alive = list(rows)
    report = []
    for rule in rules:
        key, is_score = rule_key(rule)
        op, target = rule["op"], rule["value"]
        checked_values, missing_rows, failed_rows, survivors = [], [], [], []
        for row in alive:
            value, present = row_value(row, key, is_score)
            if not present:
                missing_rows.append(row)
                if missing_policy == "keep":
                    survivors.append(row)
                continue
            checked_values.append(value)
            if rule_passes(value, op, target):
                survivors.append(row)
            else:
                failed_rows.append(row)
        removed = len(alive) - len(survivors)
        report.append({
            "rule": rule,
            "key": key,
            "description": rule_description(rule),
            "missing": len(missing_rows),
            "failed": len(failed_rows),
            "removed": removed,
            "remaining": len(survivors),
            "distribution": distribution(checked_values),
            "near_cut": sum(1 for v in checked_values if near_cut(v, op, target)),
        })
        alive = survivors
    kept_ids = [row["id"] for row in alive]
    return kept_ids, report


def print_report(report, missing_policy):
    header = f"{'rule':40s} {'removed':>8s} {'remaining':>10s}"
    print(header)
    print("-" * len(header))
    for r in report:
        print(f"{r['description']:40s} {r['removed']:>8d} {r['remaining']:>10d}")
        print(f"    missing {r['key']:<15s} {r['missing']:>8d}  (policy={missing_policy})")
        d = r["distribution"]
        if d:
            print(f"    distribution: min={d['min']:.4g}  p5={d['p5']:.4g}  p50={d['p50']:.4g}  "
                  f"p95={d['p95']:.4g}  max={d['max']:.4g}   within 5% of {r['rule']['value']}: {r['near_cut']}")
        else:
            print("    distribution: no row reached this rule with a present value")


def write_selection(kept_ids, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for rid in kept_ids:
            f.write(json.dumps(rid) + "\n")


def write_meta(meta_path, manifest_path, manifest_sha256, missing_policy, report, kept_count):
    meta = {
        "notebook": "tts_select.ipynb",
        "instruments": {},  # this notebook measures nothing; scoring is the measure notebook's job
        "input_manifest": {"path": manifest_path, "sha256": manifest_sha256},
        "missing_policy": missing_policy,
        "rules": [
            {"rule": r["rule"], "removed": r["removed"], "missing": r["missing"],
             "remaining_after": r["remaining"]}
            for r in report
        ],
        "kept": kept_count,
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    return meta


def run_selection(rows, rules, missing_policy, output_path, allow_empty,
                   manifest_path, manifest_sha256):
    kept_ids, report = select(rows, rules, missing_policy)
    removed_total = len(rows) - len(kept_ids)
    assert len(kept_ids) + removed_total == len(rows), "kept + removed must equal the input row count"
    if not kept_ids and not allow_empty:
        raise ValueError(
            f"selection would keep 0 of {len(rows)} rows; refusing to write an empty "
            f"selection. Set ALLOW_EMPTY=1 to allow this."
        )
    write_selection(kept_ids, output_path)
    meta = write_meta(f"{output_path}.meta.json", manifest_path, manifest_sha256,
                       missing_policy, report, len(kept_ids))
    return kept_ids, report, meta


print("Selection functions defined.")


## 2 &middot; Load the manifest

In [ ]:
def build_demo_manifest():
    """20 synthetic voice-conversion rows: mixed pass/fail scores, one missing score per key."""
    # id, len_diff_s, cer, echo_cos, spk_cos (None = the row lacks that score)
    demo_scores = [
        ("clip001",  0.02, 0.03, 0.10, 0.90),
        ("clip002", -0.05, 0.05, 0.20, 0.85),
        ("clip003",  0.08, 0.02, 0.15, 0.88),
        ("clip004", -0.08, 0.04, 0.30, 0.80),
        ("clip005",  0.01, 0.15, 0.10, 0.92),
        ("clip006",  0.03, 0.40, 0.05, 0.95),
        ("clip007",  0.02, 0.05, 0.90, 0.85),
        ("clip008", -0.01, 0.03, 0.99, 0.99),
        ("clip009",  0.04, 0.06, 0.20, 0.60),
        ("clip010", -0.02, 0.07, 0.25, 0.50),
        ("clip011",  0.03, 0.02, 0.15, 0.80),
        ("clip012", -0.04, 0.08, 0.30, 0.78),
        ("clip013",  0.10, 0.20, 0.90, 0.40),
        ("clip014",  0.02, None, 0.10, 0.90),
        ("clip015",  0.01, 0.04, None, 0.85),
        ("clip016",  0.03, 0.05, 0.15, None),
        ("clip017",  None, 0.03, 0.10, 0.90),
        ("clip018",  0.058, 0.03, 0.10, 0.90),
        ("clip019",  0.03, 0.096, 0.10, 0.90),
        ("clip020",  0.03, 0.03, 0.83, 0.90),
    ]
    rows = []
    for i, (rid, len_diff, cer, echo, spk) in enumerate(demo_scores):
        scores = {}
        if len_diff is not None:
            scores["len_diff_s"] = len_diff
        if cer is not None:
            scores["cer"] = cer
        if echo is not None:
            scores["echo_cos"] = echo
        if spk is not None:
            scores["spk_cos"] = spk
        rows.append({
            "id": rid,
            "text": f"demo sentence {i + 1}",
            "lang": "en-US",
            "audio": f"{rid}.wav",
            "duration_s": round(2.0 + 0.1 * i, 2),
            "sample_rate": 22050,
            "source": {"kind": "tts", "plugin": "demo", "voice": "demo", "version": "0"},
            "parent_id": f"src{i + 1:03d}",
            "identity": "miro" if i % 2 == 0 else "dii",
            "scores": scores,
        })
    return rows


if MANIFEST_PATH:
    manifest_path_used = MANIFEST_PATH
    rows = read_manifest(manifest_path_used)
else:
    demo_dir = Path("tts_select_demo")
    demo_dir.mkdir(parents=True, exist_ok=True)
    manifest_path_used = str(demo_dir / "manifest.jsonl")
    with open(manifest_path_used, "w", encoding="utf-8") as f:
        for row in build_demo_manifest():
            f.write(json.dumps(row) + "\n")
    rows = read_manifest(manifest_path_used)
    print(f"MANIFEST_PATH not set: wrote a {len(rows)}-row demo manifest to {manifest_path_used}")

manifest_sha256 = sha256_file(manifest_path_used)
print(f"Loaded {len(rows)} rows from {manifest_path_used}")
print(f"sha256: {manifest_sha256}")

if not SELECTION_PATH:
    SELECTION_PATH = str(Path(manifest_path_used).with_suffix("")) + "_selection.jsonl"
    print(f"SELECTION_PATH not set: using {SELECTION_PATH}")


## 3 &middot; Run the configured selection

Runs `RULES` against the loaded manifest and writes `SELECTION_PATH` and its `.meta.json`
sidecar. With the default `RULES=[]` every row is kept, the table below has no rows, and the
selection file holds every `id` in the manifest.


In [ ]:
kept_ids, report, meta = run_selection(
    rows, RULES, MISSING, SELECTION_PATH, ALLOW_EMPTY,
    manifest_path_used, manifest_sha256,
)

print_report(report, MISSING)
print()
print(f"kept {len(kept_ids)} of {len(rows)} rows")
print(f"selection written to {SELECTION_PATH}")
print(f"sidecar written to {SELECTION_PATH}.meta.json")


## 4 &middot; Suggested starting rules for voice-converted clips (example, not defaults)

`RULES` defaults to `[]`. The rules below are a starting point for step 8, selecting
converted clips before training &mdash; a trainer sets `RULES` to something like this
explicitly; the notebook never applies them on its own.

- **`abs(len_diff_s) <= 0.07`.** voiceclonnx's chatterbox engine adds 20 to 64 ms of audio
  per conversion, measured on 14 conversions to three reference voices; a much larger gap points at a bad conversion
  rather than that engine's normal padding.
- **`cer <= 0.10`.** The recogniser round trip should still read back close to the source
  text.
- **`echo_cos < 0.85`.** A high value means the "converted" clip is close to the identity
  reference clip itself rather than a new rendering of the source speech.
- **A `spk_cos` floor, read off this notebook's own distribution line for `spk_cos`** on the
  manifest at hand, rather than assumed from another dataset.

```json
[
  {"score": "len_diff_s", "op": "abs<=", "value": 0.07},
  {"score": "cer", "op": "<=", "value": 0.10},
  {"score": "echo_cos", "op": "<", "value": 0.85},
  {"score": "spk_cos", "op": ">=", "value": 0.75}
]
```

Section 5 below runs exactly this list, as a demonstration, against the built-in demo
manifest.


## 5 &middot; Proof: demo manifest, both `MISSING` policies, the empty-selection refusal,
and the kept-plus-removed assertion

This section always runs the demo manifest above through the example rules from section 4,
independently of `MANIFEST_PATH`/`RULES` on this run, so the counts below are reproducible.
It shows the same rules under both `MISSING` policies, the refusal to write an empty
selection, the `ALLOW_EMPTY=1` override, and one passing and one failing row per rule.


In [ ]:
import tempfile

proof_dir = Path(tempfile.mkdtemp(prefix="tts_select_proof_"))
proof_rows = build_demo_manifest()
proof_manifest_path = str(proof_dir / "manifest.jsonl")
with open(proof_manifest_path, "w", encoding="utf-8") as f:
    for row in proof_rows:
        f.write(json.dumps(row) + "\n")
proof_manifest_sha256 = sha256_file(proof_manifest_path)

EXAMPLE_RULES = [
    {"score": "len_diff_s", "op": "abs<=", "value": 0.07},
    {"score": "cer", "op": "<=", "value": 0.10},
    {"score": "echo_cos", "op": "<", "value": 0.85},
    {"score": "spk_cos", "op": ">=", "value": 0.75},
]

print(f"demo manifest: {len(proof_rows)} rows, sha256={proof_manifest_sha256}")

for policy in ("keep", "drop"):
    print(f"\n=== MISSING={policy} ===")
    out_path = str(proof_dir / f"selection_{policy}.jsonl")
    p_kept, p_report, p_meta = run_selection(
        proof_rows, EXAMPLE_RULES, policy, out_path, allow_empty=False,
        manifest_path=proof_manifest_path, manifest_sha256=proof_manifest_sha256,
    )
    print_report(p_report, policy)
    removed_total = len(proof_rows) - len(p_kept)
    assert len(p_kept) + removed_total == len(proof_rows)
    print(f"kept {len(p_kept)} + removed {removed_total} == input {len(proof_rows)}  (assertion holds)")
    print(f"kept ids: {p_kept}")


In [ ]:
print("=== empty-selection refusal: ALLOW_EMPTY=0 (default) ===")
always_fails = [{"field": "duration_s", "op": ">", "value": 999999}]
empty_out = str(proof_dir / "selection_empty.jsonl")
try:
    run_selection(proof_rows, always_fails, "keep", empty_out, allow_empty=False,
                  manifest_path=proof_manifest_path, manifest_sha256=proof_manifest_sha256)
    print("did not refuse \u2014 this would be a bug")
except ValueError as e:
    print(f"refused, as required: {e}")
print(f"file written despite the refusal? {Path(empty_out).exists()}")

print("\n=== ALLOW_EMPTY=1 lets the same rule set write an empty selection ===")
p_kept, p_report, p_meta = run_selection(
    proof_rows, always_fails, "keep", empty_out, allow_empty=True,
    manifest_path=proof_manifest_path, manifest_sha256=proof_manifest_sha256,
)
print(f"kept: {p_kept}")
print(f"selection file written: {Path(empty_out).exists()}, "
      f"sidecar written: {Path(empty_out + '.meta.json').exists()}")

print("\n=== one passing and one failing row per rule ===")
for rule in EXAMPLE_RULES:
    key, _ = rule_key(rule)
    scored = [(row["id"], row["scores"][key]) for row in proof_rows if key in row.get("scores", {})]
    passing = next((rid for rid, v in scored if rule_passes(v, rule["op"], rule["value"])), None)
    failing = next((rid for rid, v in scored if not rule_passes(v, rule["op"], rule["value"])), None)
    print(f"{rule_description(rule)}: pass={passing}  fail={failing}")
